# Plot reproducibility

This notebook uses the local `monaqa2` package and the rotation-based resource model (`FIXED_POINT`, randomized Hamiltonian grid `k=2`, 50 Trotter steps). Existing experimental datasets are required. Restart the kernel after replacing the package.

The shipped `layden` generator averages 20 mixing weights and continuous time; its gaps are not a validation of the 4-by-4 Trotterized circuit. Runtime plots using those data are resource projections. The main runtime shading represents hardware scenarios; percentile-band wrappers are available separately in `plot_main_runtime_extra`.

## Visualization settings

Figures are exported as PNG and PGF using a serif font and `pdflatex`.


In [1]:
import sys
from pathlib import Path
from pathlib import Path
import matplotlib as mpl

PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "monaqa2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or its notebook directory.")
sys.path.insert(0, str(PROJECT_ROOT))

from monaqa2.data import runtime as runtime_model
from monaqa2.data.filename import CLASSICAL_QUERY_FILE, SPECTRAL_GAP_FILE
from monaqa2.data.runtime import tight_schedule_annealing

if Path(runtime_model.__file__).resolve() != PROJECT_ROOT / "monaqa2/data/runtime.py":
    raise ImportError("A different monaqa2 is already loaded; restart the kernel.")

NORM_ONE_FILE = PROJECT_ROOT / "data/sk_upper_bound_one_norm.npy"
NORM_OP_FILE = PROJECT_ROOT / "data/sk_upper_bound_op_norm.npy"
VARIANCE_FILE = PROJECT_ROOT / "estimation_variance/var_energies.pkl"
required = (CLASSICAL_QUERY_FILE, SPECTRAL_GAP_FILE, NORM_ONE_FILE, NORM_OP_FILE, VARIANCE_FILE)
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing experiment files:\n" + "\n".join(missing))
for directory in ("plots/main_queries", "plots/main_runtime", "plots/appx"):
    Path(directory).mkdir(parents=True, exist_ok=True)

mpl.use("pgf")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
from scipy.optimize import least_squares

REGULAR_SIZE, SMALL_SIZE = 10, 8
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex", "text.usetex": True, "pgf.rcfonts": False,
    "font.family": "serif", "pgf.preamble": "",
    "font.size": REGULAR_SIZE, "axes.labelsize": REGULAR_SIZE,
    "axes.titlesize": REGULAR_SIZE, "figure.titlesize": REGULAR_SIZE,
    "xtick.labelsize": SMALL_SIZE, "ytick.labelsize": SMALL_SIZE,
    "legend.fontsize": SMALL_SIZE,
})


## Main text | Query advantage | Figure 1


In [2]:
from monaqa2.data.plot_main_queries import plot_annealing_classical_and_quantum_queries_vs_n

In [3]:
plt.close()
fig, axs = plot_annealing_classical_and_quantum_queries_vs_n(
    beta=4, 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing, 
    n_plot_min=3, 
    n_plot_max=50,
    debug=False,
    show_numerical_boundary=True,
    legend_placement="top_left",
    show_legend=True,
    show_zoom_in=True,
    zoom_xlim=(6, 10),
    zoom_ylim=(1e3, 1e4),
    zoom_bbox=(0.10, 0.45, 0.20, 0.22),
    zoom_tick_fontsize=7.0,
    x_right_padding=4.0,
    show_zoom_in_border=True,
    band_percentiles=(0, 100),
    line_width=1.5
)
filename = f"plots/main_queries/queries_miniature"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Main text | Runtime advantage | Figure 2


In [4]:
# Base runtime plots: shading shows hardware scenarios.
from monaqa2.data.plot_main_runtime import plot_annealing_classical_and_quantum_runtime_vs_n, plot_annealing_classical_and_quantum_runtime_vs_n_shadowed

In [5]:
plt.close()
fig, axs = plot_annealing_classical_and_quantum_runtime_vs_n_shadowed(
    beta=4, 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing, 
    classical_device="fpga",
    physical_error_rate=1e-3,
    physical_operation_time_min=20e-9,
    physical_measurement_time_min=200e-9,
    physical_cost_multipliers=[10, 100, 1000],
    num_trotter_steps=20,
    k_randomized_grid=2,
    arithmetic_type="FIXED_POINT",
    show_multiplier_label=True,
    statistic="mean+std",
    n_plot_min=3, 
    n_plot_max=100,
    debug=False,
    show_regime_separator=True,
    show_one_year_line=True,
    legend_placement="bottom_right",
    optimistic_pessimistic_intercept=False,
    show_direct_enumeration_line=False,
    show_legend=True,
    x_right_padding=4.0,
    y_top_padding=4.0,
    line_width=1.5,
)
filename = f"plots/main_runtime/runtime_shadow_1e-3"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Appendix A | Empirical upper bounds for the 1-norm and operator norm | Figure 3


In [6]:
from monaqa2.data.plot_aux import plot_sk_upper_bounds

fig, ax = plt.subplots(figsize=(13 / 2.54, 5.6 / 2.54))
plot_sk_upper_bounds(fig=fig, ax=ax, one_norm_file=NORM_ONE_FILE, op_norm_file=NORM_OP_FILE, output_file=None, legend_placement="right")
fig.tight_layout()

filename = "plots/appx/sk_bounds"
fig.savefig(filename + ".png", dpi=300, bbox_inches="tight")
fig.savefig(filename + ".pgf", dpi=300, bbox_inches="tight")
plt.close(fig)

## Appendix C


In [7]:
from monaqa2.data.plot_table import plot_spectral_gap_vs_beta_table
from monaqa2.data.plot_table import plot_spectral_gap_vs_n_table
from monaqa2.data.plot_table import plot_last_step_classical_queries_and_spectral_gap_vs_n_table
from monaqa2.data.plot_table import plot_last_step_classical_queries_and_quantum_queries_vs_n_table
from monaqa2.data.plot_table import plot_annealing_classical_and_quantum_queries_vs_n_table
from monaqa2.data.plot_table import plot_spectral_gap_quantiles_vs_n

### Appendix C | Spectral gap $\delta(\beta)$ | Figures 4 and 5


In [8]:
plt.close()
fig, ax = plot_spectral_gap_vs_beta_table(
    fixed_ns=[5, 6, 7, 8, 9, 10], 
    miniature_scale=1.25, 
    ncols=3, 
    statistic="mean+std",
    wspace=0.12,
    show_legend=True,
    legend_y=0,
    legend_fontsize=14,
    figsize=(33.0 / 2.54, 18 / 2.54),
    inset_bounds=(0.125, 0.100, 0.438, 0.369),
    
)
fig.tight_layout()
filename = f"plots/appx/spectral_gap_vs_beta"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [9]:
plt.close()
fig, ax = plot_spectral_gap_vs_beta_table(
    fixed_ns=[5, 6, 7, 8, 9, 10], 
    ncols=3, 
    statistic="mean+std-tail",
    wspace=0.12,
    show_legend=True,
    legend_y=0,
    legend_fontsize=14,
    figsize=(33.0 / 2.54, 18 / 2.54),
    inset_bounds=(0.125, 0.100, 0.438, 0.369)
)
fig.tight_layout()
filename = f"plots/appx/spectral_gap_vs_beta_notail"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Appendix C | Spectral gap $\delta(n)$ | Figures 6 and 7


In [10]:
plt.close()
fig, ax = plot_spectral_gap_vs_n_table(
    fixed_betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0], 
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std", 
    show_legend=True,
    y_limit_measurement_scale=1.0,
    y_limit_max_extra_orders=5.0,
    y_tick_order_step=3,
    wspace=0.12,
    legend_fontsize=12,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    extrapolation_alpha_factor=0.95,
)
filename = f"plots/appx/spectral_gap_vs_n"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [11]:
plt.close()
fig, ax = plot_spectral_gap_vs_n_table(
    fixed_betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0], 
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std-tail",
    y_limit_measurement_scale=1.0,
    y_limit_max_extra_orders=5.0,
    y_tick_order_step=3,
    wspace=0.12,
    legend_fontsize=10,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    extrapolation_alpha_factor=0.95)
filename = f"plots/appx/spectral_gap_vs_n_notail"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Appendix C | Classical query count $Q_{\rm cl,warm}(n)$ versus $\delta(n)$ | Figures 8 and 9


In [13]:
plt.close()
fig, ax, ax_gap = plot_last_step_classical_queries_and_spectral_gap_vs_n_table(
    betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0], 
    epsilon=1e-2,
    show_inverse_gap=True,
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=3.9,
)
filename = f"plots/appx/classical_queries_eps-2"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [14]:
plt.close()
fig, ax, ax_gap = plot_last_step_classical_queries_and_spectral_gap_vs_n_table(
    betas=[1.0, 1.0, 1.0, 4.0, 4.0, 4.0], 
    epsilon=[1e-2, 1e-4, 1e-8] * 2,
    show_inverse_gap=True,
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=3.9,
)
filename = f"plots/appx/classical_queries_all_epsilon"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Appendix C | Quantum versus classical query counts without annealing | Figures 10 and 11


In [15]:
plt.close()
fig, ax = plot_last_step_classical_queries_and_quantum_queries_vs_n_table(
    betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0], 
    epsilon=1e-2,
    ncols=3, 
    n_plot_min=4, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=3.9,
)
filename = "plots/appx/classical_and_quantum_queries_eps-2"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [16]:
plt.close()
fig, ax = plot_last_step_classical_queries_and_quantum_queries_vs_n_table(
    betas=[1.0, 1.0, 1.0, 4.0, 4.0, 4.0], 
    epsilon=[1e-2, 1e-4, 1e-8] * 2,
    ncols=3, 
    n_plot_min=4, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=3.9,
    dashed_classical_query_line=True,
)
filename = "plots/appx/classical_and_quantum_queries_all_epsilon"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Appendix C | Quantum versus classical query counts with annealing | Figure 12


In [17]:
plt.close()
fig, ax = plot_annealing_classical_and_quantum_queries_vs_n_table(
    betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0], 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing,
    ncols=3, 
    n_plot_min=4, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=3.9,
    dashed_classical_query_line=True,
    show_schedule_vertical_lines=False
)
filename = "plots/appx/classical_and_quantum_queries_full_annealing"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Appendix C | Spectral gap of the randomized proposal at $\beta = 4$ by $n$ and for different grid sizes | Figure 13

Taken from notebook 3.4.

In [28]:
from monaqa2.data.plot_aux import plot_spectral_gaps_randomized_subset_parameters
from monaqa2.data.filename import CONVERGENCE_RANDOMIZED_HYPERPARAMS_FILE, GRIDSEARCH_RANDOMIZED_HYPERPARAMS_FILE


with open(CONVERGENCE_RANDOMIZED_HYPERPARAMS_FILE) as f:
    spectral_gaps = json.load(f)
    plot_spectral_gaps_randomized_subset_parameters(spectral_gaps, is_transition=True)
    filename = "plots/appx/2_randomized_grid_params_spectral_gaps"
    plt.savefig(filename + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(filename + ".pgf", dpi=300, bbox_inches="tight")

### Appendix C | Impact of a single choice of $(t, \gamma)$ for the spectral gap of the randomized proposal at $\beta = 4$ by $n$ and for different grid sizes | Figure 14

In [33]:
from monaqa2.data.filename import CONVERGENCE_RANDOMIZED_HYPERPARAMS_FILE, GRIDSEARCH_RANDOMIZED_HYPERPARAMS_FILE
from monaqa2.data.plot_aux import plot_randomized_grid_search_difference

grid_search_file = Path(GRIDSEARCH_RANDOMIZED_HYPERPARAMS_FILE)
averaged_matrix_file = Path(CONVERGENCE_RANDOMIZED_HYPERPARAMS_FILE)

filename = "plots/appx/3_randomized_grid_search_difference"
for ext in (".png", ".pgf"):
    plot_randomized_grid_search_difference(grid_search_file, averaged_matrix_file, k=3, output_file=filename+ext)

### Appendix C | Spectral gap dependence on the Trotter steps | Figure 15 and 16

See notebook robustness_trotterization/trotter_scaling_n.ipynb



### Appendix C | Impact of normalization on the spectral gap | Figure 17 and 18

See notebook robustness_model_normalization/model_normalization.ipynb.

### Appendix C | Spectral-gap quantiles versus $n$ | Figure 19

Median and percentile bands across all 100 instances at $\beta=4$, with one panel per proposal.

In [3]:
plt.close()
fig, axs = plot_spectral_gap_quantiles_vs_n(
    beta=4.0,
    in_file=SPECTRAL_GAP_FILE,
    n_min=3,
    n_max=10,
    expected_instances=100,
)
Path("plots").mkdir(exist_ok=True)
filename = "plots/appx/spectral_gap_quantiles_vs_n"
fig.savefig(filename + ".png", dpi=300, bbox_inches="tight")
fig.savefig(filename + ".pgf", dpi=300, bbox_inches="tight")

## Appendix D | Comparison of classical-device runtimes | Figure 20

In [18]:
from monaqa2.data.plot_main_runtime import plot_annealing_classical_runtime_vs_n

plt.close()
fig, ax = plot_annealing_classical_runtime_vs_n(
    beta=4.0,
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing,
    n_plot_min=4,
    n_plot_max=100,
    n_fit_min=5,
    n_fit_max=10,
    statistic="mean+std",
    show_legend=True,
    legend_placement="out",
    legend_y_shift=-0.30,
    line_width=1.8,
    line_alpha=0.95,
    marker_size=3.2,
    gpu_markevery=4,
    grid_color="0.92",
    grid_linewidth=0.55,
    show_regime_separator=True,
    regime_separator_n=10.0,
)

filename = "plots/appx/classical_annealing_runtime_by_device"
plt.savefig(filename + ".png", dpi=300, bbox_inches="tight")
plt.savefig(filename + ".pgf", dpi=300, bbox_inches="tight")

## Appendix F | Circuits | Figure 21

Done with quantikz and exported to PDF

## Appendix G | Extensibility of the SK model and energy variance | Figure 22 (non-fitted variant)


In [11]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.text import Text

numerical_var_energies = pd.read_pickle(VARIANCE_FILE)
df = numerical_var_energies.copy()
df = df.loc[np.isfinite(df["var_energy"]) & (df["var_energy"] > 0)].copy()
df["V_over_n"] = df["var_energy"] / df["n"]

agg = df.groupby(["n", "beta"], as_index=False).agg(V=("var_energy", "mean"), V_std=("var_energy", "std"), count=("var_energy", "count"))

beta_ticks = [0.0, 1.0 / 32.0, 1.0 / 16.0, 3.0 / 32.0, 1.0 / 8.0, 3.0 / 16.0, 1.0 / 4.0, 3.0 / 8.0, 1.0 / 2.0, 5.0 / 8.0, 3.0 / 4.0, 13.0 / 16.0, 7.0 / 8.0, 15.0 / 16.0, 1.0, 9.0 / 8.0, 5.0 / 4.0, 11.0 / 8.0, 3.0 / 2.0, 7.0 / 4.0, 2.0, 3.0, 4.0, 6.0, 8.0, 16.0]
beta_tick_labels = [r"$0$", r"$1/32$", r"$1/16$", r"$3/32$", r"$1/8$", r"$3/16$", r"$1/4$", r"$3/8$", r"$1/2$", r"$5/8$", r"$3/4$", r"$13/16$", r"$7/8$", r"$15/16$", r"$1$", r"$9/8$", r"$5/4$", r"$11/8$", r"$3/2$", r"$7/4$", r"$2$", r"$3$", r"$4$", r"$6$", r"$8$", r"$16$"]

cmap = mpl.colors.LinearSegmentedColormap.from_list("blue_to_orange", ["tab:blue", "tab:orange"])
norm = mpl.colors.Normalize(vmin=df["n"].min(), vmax=df["n"].max())
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

mean_by_n_beta = df.groupby(["n", "beta"], as_index=False)["V_over_n"].mean()
mean_by_beta = mean_by_n_beta.groupby("beta", as_index=False)["V_over_n"].mean()

plt.close("all")
fig, ax = plt.subplots(figsize=(14, 4.8))

for n, group in df.groupby("n"):
    ax.scatter(group["beta"], group["V_over_n"], s=10, alpha=0.18, color=cmap(norm(n)))

ax.plot(mean_by_beta["beta"], mean_by_beta["V_over_n"], color="black", linewidth=2.8, label="average")
ax.set_xscale("symlog", linthresh=1.0)
ax.set_xlim(0, 16)
ax.set_xticks(beta_ticks)
ax.set_xticklabels(beta_tick_labels, rotation=60, ha="right")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"$\sigma^2(n,\beta)/n$")
ax.set_title(r"Energy-variance density $\sigma^2(n,\beta)/n$")
fig.colorbar(sm, ax=ax, label=r"$n$")
ax.legend()
fig.tight_layout()

filename = "plots/appx/annealing_extensibility"
#fig.savefig(filename + ".pgf", dpi=300)

for text in fig.findobj(match=Text):
    text.set_usetex(False)

with mpl.rc_context({"text.usetex": False}):
    pass #fig.savefig(filename + ".png", backend="agg", dpi=300, bbox_inches="tight")

## Appendix G | Extensibility of the SK model and energy variance | Figure 22 (fitted variant)


In [25]:
def log_rmse(y, yhat):
    return np.sqrt(np.mean((np.log(yhat) - np.log(y)) ** 2))


def fit_data(agg):
    d = agg.loc[(agg["V"] > 0) & np.isfinite(agg["V"])].copy()
    n = d["n"].to_numpy(dtype=float)
    beta = d["beta"].to_numpy(dtype=float)
    y = np.log(d["V"].to_numpy(dtype=float))
    return n, beta, y


def v_ansatz_reduced(theta, beta):
    log_v1, kappa, log_tau = theta
    v1 = np.exp(log_v1)
    tau = np.exp(log_tau)
    beta = np.asarray(beta, dtype=float)
    v = np.empty_like(beta, dtype=float)
    mask = beta < 1.0
    v[mask] = np.exp(log_v1 * beta[mask] + kappa * beta[mask] * (1.0 - beta[mask]))
    v[~mask] = v1 * (1.0 + (beta[~mask] - 1.0) / tau) ** (-3.5)
    return v


def V_ansatz_reduced(theta, n, beta):
    return np.asarray(n, dtype=float) * v_ansatz_reduced(theta, beta)


def fit_ansatz_reduced(agg):
    d = agg.loc[(agg["V"] > 0) & np.isfinite(agg["V"])].copy()
    n = d["n"].to_numpy(dtype=float)
    beta = d["beta"].to_numpy(dtype=float)
    y = np.log(d["V"].to_numpy(dtype=float))

    def residual(theta):
        return np.log(V_ansatz_reduced(theta, n, beta)) - y

    theta0 = [np.log(0.36), 1.25, np.log(1.3)]
    res = least_squares(residual, theta0)
    return res.x, res.cost


theta_reduced, cost_reduced = fit_ansatz_reduced(agg)
pred_reduced = V_ansatz_reduced(theta_reduced, agg["n"].to_numpy(dtype=float), agg["beta"].to_numpy(dtype=float))

log_v1, kappa, log_tau = theta_reduced

print("cost =", cost_reduced)
print("log-RMSE =", log_rmse(agg["V"].to_numpy(dtype=float), pred_reduced))
print("v0 =", 1.0)
print("v1 =", np.exp(log_v1))
print("kappa =", kappa)
print("tau =", np.exp(log_tau))
print("q =", 3.5)

beta_plot = np.linspace(0.0, 16.0, 2000)
v_plot = v_ansatz_reduced(theta_reduced, beta_plot)

mean_by_n_beta = df.groupby(["n", "beta"], as_index=False)["V_over_n"].mean()
mean_by_beta = mean_by_n_beta.groupby("beta", as_index=False)["V_over_n"].mean()



plt.close("all")
LABEL_SIZE_BIG = 14
LABEL_SIZE_SMALL = 10
fig, ax = plt.subplots(figsize=(14, 4.8))

for n, group in df.groupby("n"):
    ax.scatter(group["beta"], group["V_over_n"], s=10, alpha=0.18, color=cmap(norm(n)))

ax.plot(mean_by_beta["beta"], mean_by_beta["V_over_n"], color="black", linewidth=2.8, label="average")
ax.plot(beta_plot, v_plot, color="red", linewidth=2.8, linestyle="--", label=r"fit: $v_0=1$, $q=7/2$", zorder=4)

ax.set_xscale("symlog", linthresh=1.0)
ax.set_xlim(0, 16)
ax.set_xticks(beta_ticks)
ax.set_xticklabels(beta_tick_labels, rotation=60, ha="right", fontsize=LABEL_SIZE_SMALL)
ax.tick_params(axis="y", labelsize=LABEL_SIZE_SMALL)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel(r"$\beta$", fontsize=LABEL_SIZE_BIG)
ax.set_ylabel(r"$\sigma^2(n,\beta)/n$", fontsize=LABEL_SIZE_BIG)
ax.set_title(r"Energy-variance density $\sigma^2(n,\beta)/n$", fontsize=LABEL_SIZE_BIG)
fig.colorbar(sm, ax=ax, label=r"$n$")
ax.legend(fontsize=12)
fig.tight_layout()

for ax in fig.axes:
    for artist in (*ax.collections, *ax.lines):
        artist.set_rasterized(True)

filename = "plots/appx/annealing_extensibility_fitted"
fig.savefig(filename + ".pgf", dpi=300)
fig.savefig(filename + ".pdf", backend="pgf", dpi=300)
plt.close(fig)

cost = 2.8614770088953287
log-RMSE = 0.08565699350099236
v0 = 1.0
v1 = 0.3710820541889474
kappa = 1.216378250989564
tau = 1.2827403319613835
q = 3.5


## Appendix H | Surface-code distance | Figures 23 and 24


In [21]:
from monaqa2.data.plot_table import plot_annealing_classical_and_quantum_surface_code_distance_vs_n_table

In [22]:
plt.close()

fig, axs = plot_annealing_classical_and_quantum_surface_code_distance_vs_n_table(
    betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0],
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing,

    # Table layout
    ncols=3,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    hspace=0.56,
    wspace=0.12,

    # Fitting and plotted range
    n_fit_min=5,
    n_fit_max=10,
    n_plot_min=4,
    n_plot_max=100,
    statistic="mean+std",

    # Surface-code model
    physical_error_rate=1e-3,

    # Quantum circuit model
    num_trotter_steps=20,
    arithmetic_type="FIXED_POINT",

    # Plot appearance
    show_legend=False,
    line_width=2.0,
    line_alpha=0.95,
    rowwise_y_limits=True,

    # Diagnostics
    debug=False,
)

for ax in np.asarray(axs).flat:
    ax.tick_params(
        axis="y",
        which="both",
        right=False,
        labelright=False,
    )

filename = "plots/appx/surface_code_distance_full_annealing_beta"
fig.savefig(filename + ".pgf", dpi=300)
fig.savefig(filename + ".png", dpi=300)

In [23]:
plt.close()

fig, axs = plot_annealing_classical_and_quantum_surface_code_distance_vs_n_table(
    betas=[1.0, 1.0, 1.0, 4.0, 4.0, 4.0], 
    epsilon=[1e-2, 1e-4, 1e-8] * 2,
    annealing_schedule_generator=tight_schedule_annealing,

    # Table layout
    ncols=3,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    hspace=0.56,
    wspace=0.12,

    # Fitting and plotted range
    n_fit_min=5,
    n_fit_max=10,
    n_plot_min=4,
    n_plot_max=100,
    statistic="mean+std",

    # Surface-code model
    physical_error_rate=1e-3,

    # Quantum circuit model
    num_trotter_steps=20,
    arithmetic_type="FIXED_POINT",

    # Plot appearance
    show_legend=False,
    line_width=2.0,
    line_alpha=0.95,
    rowwise_y_limits=True,

    # Diagnostics
    debug=False,
)

for ax in np.asarray(axs).flat:
    ax.tick_params(
        axis="y",
        which="both",
        right=False,
        labelright=False,
    )

filename = "plots/appx/surface_code_distance_full_annealing_eps"
fig.savefig(filename + ".pgf", dpi=300)
fig.savefig(filename + ".png", dpi=300)

## Appendix H | Runtime plots | Figures 25 and 26


In [24]:
from monaqa2.data.plot_table import (
    plot_annealing_classical_and_quantum_runtime_vs_n_table,
    plot_annealing_classical_and_quantum_runtime_vs_n_table_shadowed
)

In [25]:
plt.close()

fig, axs = plot_annealing_classical_and_quantum_runtime_vs_n_table_shadowed(
    betas=[1.0, 2.0, 4.0, 10.0, 20.0, 40.0],
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing,

    # Table layout
    ncols=3,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    hspace=0.56,
    wspace=0.12,
    last_row_legend_y=-0.18,

    # Fitting and plotted range
    n_fit_min=5,
    n_fit_max=10,
    n_plot_min=4,
    n_plot_max=100,
    statistic="mean+std",

    # Classical hardware
    classical_device="fpga",

    # Quantum hardware range
    physical_error_rate=1e-3,
    physical_operation_time_min=20e-9,
    physical_measurement_time_min=200e-9,
    physical_cost_multipliers=[1000],
    show_multiplier_label=True,

    # Quantum circuit model
    num_trotter_steps=20,
    arithmetic_type="FIXED_POINT",

    # Plot appearance
    mode="full",
    show_legend=False,
    legend_placement="legend_out",
    remove_local_beta_threshold=3.9,
    line_width=2.0,
    line_alpha=0.95,

    # Runtime axis
    runtime_ymin_seconds=1.0,
    runtime_ymax_years=1000.0,
    show_time_reference_lines=True,

    # Diagnostics
    debug=False,
)
for ax in np.asarray(axs).flat:
    ax.tick_params(
        axis="y",
        which="both",
        right=False,
        labelright=False,
    )
filename = "plots/appx/classical_and_quantum_runtime_full_annealing_shadowed_beta"
plt.savefig(filename + ".pgf", dpi=300)
plt.savefig(filename + ".png", dpi=300)

In [26]:
plt.close()

fig, axs = plot_annealing_classical_and_quantum_runtime_vs_n_table_shadowed(
    betas=[1.0, 1.0, 1.0, 4.0, 4.0, 4.0], 
    epsilon=[1e-2, 1e-4, 1e-8] * 2,
    annealing_schedule_generator=tight_schedule_annealing,

    # Table layout
    ncols=3,
    figsize=(33.0 / 2.54, 18.0 / 2.54),
    hspace=0.56,
    wspace=0.12,
    last_row_legend_y=-0.18,

    # Fitting and plotted range
    n_fit_min=5,
    n_fit_max=10,
    n_plot_min=4,
    n_plot_max=100,
    statistic="mean+std",

    # Classical hardware
    classical_device="fpga",

    # Quantum hardware range
    physical_error_rate=1e-3,
    physical_operation_time_min=20e-9,
    physical_measurement_time_min=200e-9,
    physical_cost_multipliers=[1000],
    show_multiplier_label=True,

    # Quantum circuit model
    num_trotter_steps=20,
    arithmetic_type="FIXED_POINT",

    # Plot appearance
    mode="full",
    show_legend=False,
    legend_placement="legend_out",
    remove_local_beta_threshold=3.9,
    line_width=2.0,
    line_alpha=0.95,

    # Runtime axis
    runtime_ymin_seconds=1.0,
    runtime_ymax_years=1000.0,
    show_time_reference_lines=True,

    # Diagnostics
    debug=False,
)
for ax in np.asarray(axs).flat:
    ax.tick_params(
        axis="y",
        which="both",
        right=False,
        labelright=False,
    )
filename = "plots/appx/classical_and_quantum_runtime_full_annealing_shadowed_eps"
plt.savefig(filename + ".pgf", dpi=300)
plt.savefig(filename + ".png", dpi=300)